# Save responses for eval on TreeRAG's most optimal configuration
input question file, questions.json, is shared with qms_search

In [1]:
%pip install ollama numpy pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, json, re, time, math, hashlib, textwrap
from pathlib import Path
from dataclasses import dataclass, field, asdict, replace
from typing import List, Dict, Any, Optional
from collections import Counter
from itertools import product
import ollama
import numpy as np
import pandas as pd

OLLAMA_URL  = "http://localhost:11528"
AGENT_MODEL = "gpt-oss:120b"
EMBED_MODEL = "nomic-embed-text"
CACHE_DIR  = Path("tree_cache"); TREE_FILE = CACHE_DIR / "corpus_tree.json"
QUESTIONS_FILE = "questions_updated.json"   
ANSWERS_FILE   = "treerag_answers.json"    
MAX_EVIDENCE  = 12
SCORE_MODE    = "judge"
KEEP_ALIVE    = "30m"
print(f"agent {AGENT_MODEL}; questions {QUESTIONS_FILE}; output {ANSWERS_FILE}")

agent gpt-oss:120b; questions questions_updated.json; output treerag_answers.json


In [3]:
client = ollama.Client(host=OLLAMA_URL, timeout=600)

def _model_names(r):
    raw = r.get("models", []) if hasattr(r, "get") else getattr(r, "models", [])
    out=[]
    for m in raw:
        n = getattr(m,"model",None) or getattr(m,"name",None)
        if n is None and isinstance(m,dict): n=m.get("model") or m.get("name")
        if n: out.append(n)
    return out

def _wait_until_ready():
    announced=False
    while True:
        try:
            names=_model_names(client.list())
            if any(AGENT_MODEL in n for n in names):
                if not any(EMBED_MODEL in n for n in names):
                    print(f"note: {EMBED_MODEL} not present; cluster strategy will fall back to chunk")
                print(f"ollama ok; {AGENT_MODEL} is loaded"); return
            reason=f"{AGENT_MODEL} not loaded yet"
        except Exception as e:
            reason=f"server unreachable; {type(e).__name__}: {e}"
        if not announced:
            print(f"waiting for ollama, {reason}; rechecking every 10s and wont stop"); announced=True
        time.sleep(10)

def llm(prompt, cfg, counter, num_predict=None, temperature=0, think=None):
    use_think = cfg.thinking if think is None else think
    opts={"temperature":temperature, "num_predict": num_predict or cfg.decision_cap}
    attempt=0; pass_think=True
    while True:
        kw=dict(model=AGENT_MODEL, messages=[{"role":"user","content":prompt}],
                options=opts, keep_alive=KEEP_ALIVE)
        if pass_think: kw["think"]=use_think
        try:
            r=client.chat(**kw)
            counter.calls+=1
            try: counter.in_tok  += int(r["prompt_eval_count"] or 0)
            except Exception: pass
            try: counter.out_tok += int(r["eval_count"] or 0)
            except Exception: pass
            txt=(r["message"]["content"] or "").strip()
            if not txt:
                try: txt=(r["message"]["thinking"] or "").strip()
                except Exception: pass
            return txt
        except TypeError:
            pass_think=False
        except Exception as e:
            attempt+=1
            if attempt==1 or attempt%5==0: print(f"[waiting for ollama] {type(e).__name__}: {e}; retrying")
            time.sleep(min(60, 5*2**min(attempt-1,4)))

def embed(text, counter):
    try:
        r=client.embeddings(model=EMBED_MODEL, prompt=text or " ")
        try: counter.in_tok += int(r.get("prompt_eval_count",0) or 0)
        except Exception: pass
        return r["embedding"]
    except Exception:
        return None

_wait_until_ready()
print("llm and embed helpers ready")

ollama ok; gpt-oss:120b is loaded
llm and embed helpers ready


In [ ]:
import re, math, hashlib
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Optional
from collections import Counter
import numpy as np


@dataclass
class TreeNode:
    node_id:str; node_type:str; name:str; path:str; summary:str
    content:str=""; children:List["TreeNode"]=field(default_factory=list)
    metadata:Dict[str,Any]=field(default_factory=dict)

    @classmethod
    def from_dict(cls,d):
        n=cls(node_id=d["node_id"],node_type=d["node_type"],name=d["name"],
              path=d.get("path",""),summary=d.get("summary",""),
              content=d.get("content",""),metadata=d.get("metadata",{}))
        n.children=[cls.from_dict(c) for c in d.get("children",[])]
        return n

    def is_leaf(self): return self.node_type=="chunk"
    def count_leaves(self): return 1 if self.is_leaf() else sum(c.count_leaves() for c in self.children)


@dataclass
class Question:
    qid:str; stem:str; options:Dict[str,str]; answer:str; difficulty:str=""
    answers:List[str]=field(default_factory=list)  

def _ans_list(q):
    return list(q.answers) if getattr(q,"answers",None) else ([q.answer] if q.answer else [])

def _opts_text(q):
    return "\n".join(f"{L}. {q.options[L]}" for L in "ABCDEF" if L in q.options)

def _gold_text(q):
    al=_ans_list(q)
    return "\n".join(f"{L}. {q.options.get(L,'')}" for L in al) if al else ""


@dataclass(frozen=True)
class Config:
    strategy:str="cluster"         
    working_memory:bool=True       
    backtracking:bool=True         
    thinking:bool=False            
    group_summary:str="heuristic"  
    max_branch:int=6               
    decision_cap:int=512            
    answer_cap:int=8                
    nav_includes_options:bool=False 
    breadcrumb:bool=False           
    vote_samples:int=1              
    max_steps:int=24               


# what makes this config differ from the plain default; used as a readable label
def config_name(cfg):
    base=Config()
    diffs=[f"{k}={getattr(cfg,k)}" for k in cfg.__dataclass_fields__ if getattr(cfg,k)!=getattr(base,k)]
    return "baseline" if not diffs else ", ".join(diffs)

def config_key(cfg):
    return hashlib.md5(repr(asdict(cfg)).encode()).hexdigest()[:12]


class Counters:
    def __init__(self): self.in_tok=0; self.out_tok=0; self.calls=0


def clip(t,n):
    t=re.sub(r"\s+"," ",t or "").strip()
    return t if len(t)<=n else t[:n]+" …"


# only place embeddings are touched and only to organise groups
_nav_cache={}; _embed_cache={}; _gsum_cache={}

def reset_vcaches():
    _nav_cache.clear(); _gsum_cache.clear()   # embeddings persist; they dont depend on the config

def _embed_node(node,counter):
    if node.node_id in _embed_cache: return _embed_cache[node.node_id]
    v=embed((node.name+". "+(node.summary or ""))[:2000],counter)
    _embed_cache[node.node_id]=v; return v

# tiny kmeans on unit vectors so its cosine ish
def _kmeans(vectors,k,iters=25,seed=0):
    X=np.asarray(vectors,dtype=float); n=len(X); k=max(1,min(k,n))
    X=X/np.clip(np.linalg.norm(X,axis=1,keepdims=True),1e-9,None)
    rng=np.random.default_rng(seed); C=X[rng.choice(n,size=k,replace=False)].copy()
    labels=np.full(n,-1)
    for _ in range(iters):
        d=((X[:,None,:]-C[None,:,:])**2).sum(-1); new=d.argmin(1)
        if np.array_equal(new,labels): break
        labels=new
        for j in range(k):
            pts=X[labels==j]; C[j]=pts.mean(0) if len(pts) else X[rng.integers(n)]
    return labels.tolist()

def _group_summary(members,label,cfg,counter):
    key=(label,cfg.group_summary,tuple(m.node_id for m in members))
    if key in _gsum_cache: return _gsum_cache[key]
    if cfg.group_summary=="llm":
        joined="\n".join(f"- {m.summary}" for m in members)
        prompt=("in 2-4 sentences say what this group called '"+label+"' covers so an agent can "
                "decide whether to explore it; list the main topics. respond with only the text.\n\n"+joined)
        s=llm(prompt,cfg,counter,num_predict=300)
    else:
        lines=[f"- {m.name}: {clip(m.summary,160)}" for m in members[:cfg.max_branch]]
        more=f"\n- …and {len(members)-cfg.max_branch} more" if len(members)>cfg.max_branch else ""
        s=f"a group of {len(members)} related items:\n"+"\n".join(lines)+more
    _gsum_cache[key]=s; return s

def _vid(pid,tag): return "v"+hashlib.md5(f"{pid}|{tag}".encode()).hexdigest()[:11]

def _make_vgroup(parent,members,idx,cfg,counter):
    return TreeNode(node_id=_vid(parent.node_id,f"{cfg.strategy}:{cfg.max_branch}:{idx}"),
                    node_type="vgroup",name=f"[group {idx+1} · {len(members)} items]",path="",
                    summary=_group_summary(members,f"{parent.name} group {idx+1}",cfg,counter),
                    children=list(members),metadata={"virtual":True})

def _chunk_groups(parent,kids,cfg,counter):
    size=math.ceil(len(kids)/cfg.max_branch)             
    groups=[kids[i:i+size] for i in range(0,len(kids),size)]
    return [_make_vgroup(parent,g,i,cfg,counter) for i,g in enumerate(groups)]

def _cluster_groups(parent,kids,cfg,counter):
    vecs=[_embed_node(k,counter) for k in kids]
    if any(v is None for v in vecs): return _chunk_groups(parent,kids,cfg,counter)  # no embeds; fall back
    k=max(2,min(cfg.max_branch,math.ceil(len(kids)/max(2,cfg.max_branch-2))))
    labels=_kmeans(vecs,k); buckets={}
    for kid,lab in zip(kids,labels): buckets.setdefault(lab,[]).append(kid)
    if len(buckets)<2 or max(len(v) for v in buckets.values())==len(kids):
        return _chunk_groups(parent,kids,cfg,counter)     
    ordered=[buckets[lab] for lab in sorted(buckets)]
    return [_make_vgroup(parent,m,i,cfg,counter) for i,m in enumerate(ordered)]

# the children the agent chooses among, virtualised down to max_branch
def get_nav_children(node,cfg,counter):
    key=(node.node_id,cfg.strategy,cfg.max_branch,cfg.group_summary)
    if key in _nav_cache: return _nav_cache[key]
    kids=node.children
    if cfg.strategy=="none" or len(kids)<=cfg.max_branch: res=list(kids)
    elif cfg.strategy=="chunk": res=_chunk_groups(node,kids,cfg,counter)
    elif cfg.strategy=="cluster": res=_cluster_groups(node,kids,cfg,counter)
    else: res=list(kids)
    _nav_cache[key]=res; return res


# agent
import json as _json

def _add_memory(mem,fact,cap=20):
    fact=clip(fact,500)
    if fact and fact.lower() not in ("","none","n/a") and fact not in mem:
        mem.append(fact); del mem[:-cap]

# pull a single action out of the model json, tolerant of junk
def _parse_decision(raw,n_opts,wm):
    s=re.sub(r"^```(?:json)?|```$","",raw.strip(),flags=re.M).strip()
    m=re.search(r"\{.*\}",s,flags=re.S)
    if m:
        try:
            d=_json.loads(m.group(0)); act=str(d.get("action","")).lower().strip()
            if act not in ("descend","answer","backtrack"): act="descend" if n_opts else "answer"
            ci=d.get("child",None)
            try: ci=int(ci)
            except (TypeError,ValueError): ci=None
            return {"action":act,"child":ci,"remember":(d.get("remember") or "").strip() if wm else ""}
        except Exception: pass
    return {"action":"descend" if n_opts else "backtrack","child":0 if n_opts else None,"remember":""}

def _ask(query,node,options,memory,cfg,counter,can_back):
    opts_txt="\n".join(f"[{i}] {o.name} — {clip(o.summary,360)}" for i,o in enumerate(options)) or "(no options here)"
    acts=(['"descend"'] if options else [])+['"answer"']+(['"backtrack"'] if can_back else [])
    mem_block=""; remember_field=""
    if cfg.working_memory:
        mem_block="WORKING MEMORY (facts gathered so far):\n"+("\n".join("- "+m for m in memory) or "(empty)")+"\n\n"
        remember_field='"remember": "<a useful fact from the CURRENT summary to keep, else empty>", '
    prompt=(
        "you are an agent navigating a tree of document summaries to answer a question. "
        "you see only summaries and move one node at a time.\n\n"
        f"QUESTION: {query}\n\n{mem_block}"
        f"CURRENT NODE: {node.name} [{node.node_type}]\nCURRENT SUMMARY: {clip(node.summary,900)}\n\n"
        f"CHILD OPTIONS:\n{opts_txt}\n\n"
        f"choose ONE action ({', '.join(acts)}). reply with ONLY json:\n"
        '{"reasoning":"<1 sentence>", '+remember_field+
        '"action":"descend|answer|backtrack", "child":<index or null>}\n'
        "descend into the option most likely to lead to the answer; answer if you have enough; "
        "backtrack if none of these are relevant.")
    cap=cfg.decision_cap if not cfg.thinking else max(cfg.decision_cap,1024)
    return _parse_decision(llm(prompt,cfg,counter,num_predict=cap),len(options),cfg.working_memory)

SCORE_MODE = globals().get("SCORE_MODE","judge")   # judge: free response graded 0-1 by an llm; letter: pick A-F

# a short label for the compact path string
def _short(n):
    if n.metadata.get("virtual"): return "[grp]"
    return (n.name or "?")[:22]

def _src(n): return n.metadata.get("source_file") or n.path or n.name

# run the whole traversal for one question; live draws the path on a single growing line
def run_agent(q,cfg,counter,trace=True,live=False):
    ink=(lambda s: print(s,end="",flush=True)) if live else (lambda s: None)
    nav_q=q.stem                                    # the agent only ever sees the question, never the choices
    if cfg.nav_includes_options:
        nav_q+="\noptions: "+"; ".join(f"{L}) {q.options[L]}" for L in "ABCDEF" if L in q.options)
    memory=[]; evidence=[]; seen=set(); visited={ROOT.node_id}; crumbs=[]
    trail=["root"]; backtracks=0
    stack=[{"node":ROOT,"options":get_nav_children(ROOT,cfg,counter),"tried":set()}]
    ink("  path: root")
    steps=0
    while stack and steps<cfg.max_steps:
        steps+=1; fr=stack[-1]; node=fr["node"]
        present=[(i,o) for i,o in enumerate(fr["options"]) if i not in fr["tried"] and o.node_id not in visited]
        opts=[o for _,o in present]
        can_back=cfg.backtracking and len(stack)>1
        dec=_ask(nav_q,node,opts,memory,cfg,counter,can_back)
        if cfg.working_memory and dec["remember"]: _add_memory(memory,dec["remember"])
        if cfg.breadcrumb: crumbs.append(node.summary)
        act=dec["action"]
        if act=="answer" and not evidence and opts: act="descend"   # dont answer before reading a real document
        if act=="descend" and not opts:                             # nothing left here; go back up or stop
            act="backtrack" if can_back else "answer"
        if act=="answer" or (act=="backtrack" and not can_back):    # disallowed backtrack becomes answer
            if node.node_id not in seen: evidence.append(node); seen.add(node.node_id)
            ink(" ⇒ answer"); break
        if act=="backtrack":
            popped=stack.pop(); parent=stack[-1]; backtracks+=1; trail.append("↩"); ink(" ↩")
            for i,o in enumerate(parent["options"]):
                if o.node_id==popped["node"].node_id: parent["tried"].add(i); break
            continue
        ci=dec["child"]
        if ci is None or not (0<=ci<len(opts)): ci=0    # bad index; just take the first
        child=opts[ci]; visited.add(child.node_id)
        child_opts=get_nav_children(child,cfg,counter)
        if child.is_leaf() or not child_opts:           # reached a real leaf; grab its text
            if child.node_id not in seen: evidence.append(child); seen.add(child.node_id)
            if cfg.working_memory and child.content: _add_memory(memory,clip(child.content,600))
            trail.append(_short(child)+"*"); ink(f" → {child.name}✓")
            for i,o in enumerate(fr["options"]):
                if o.node_id==child.node_id: fr["tried"].add(i); break
            continue
        trail.append(_short(child)); ink(f" → {child.name}")
        stack.append({"node":child,"options":child_opts,"tried":set()})
    if live: print()                                    # close the path line
    if SCORE_MODE=="letter":
        response=_answer_letter(q,memory,evidence,crumbs,cfg,counter)
    else:
        response=_answer_response(q,memory,evidence,crumbs,cfg,counter)
    return {"response":response,"evidence":evidence,"steps":steps,
            "backtracks":backtracks,"path":" › ".join(trail)}

def _parse_letter(text):
    t=(text or "").strip().upper()
    if not t: return ""
    cues=re.findall(r"ANSWER[^A-F]{0,8}?\b([A-F])\b",t)   # answer is B, correct answer: C; take the last
    if cues: return cues[-1]
    m=re.search(r"\b([ABCDEF])\b",t) or re.search(r"([ABCDEF])",t)
    return m.group(1) if m else ""

# build the final multiple choice prompt from whatever was gathered, then vote if asked
def _answer_letter(q,memory,evidence,crumbs,cfg,counter):
    opts=_opts_text(q)
    parts=[]
    if cfg.working_memory and memory: parts.append("notes you gathered:\n"+"\n".join("- "+m for m in memory))
    if cfg.breadcrumb and crumbs: parts.append("summaries along your path:\n"+"\n".join("- "+clip(c,200) for c in crumbs[-8:]))
    if evidence:
        ev="\n\n".join(f"[{e.metadata.get('source_file') or e.path or e.name}] {clip(e.content or e.summary,1200)}"
                       for e in evidence[:MAX_EVIDENCE])
        parts.append("evidence:\n"+ev)
    ctx="\n\n".join(parts) or "(no context gathered)"
    prompt=(f"answer the multiple choice question using the gathered information.\n\n"
            f"QUESTION: {q.stem}\nOPTIONS:\n{opts}\n\n{ctx}\n\n"
            "respond with ONLY the letter(s) of the correct option(s).")
    cap=1024 if cfg.thinking else 64
    if cfg.vote_samples<=1:
        return _parse_letter(llm(prompt,cfg,counter,num_predict=cap,temperature=0))
    votes=[_parse_letter(llm(prompt,cfg,counter,num_predict=cap,temperature=0.7)) for _ in range(cfg.vote_samples)]
    votes=[v for v in votes if v]
    return Counter(votes).most_common(1)[0][0] if votes else ""

# free text answer from whatever the traversal gathered; this is what the judge grades
def _answer_response(q,memory,evidence,crumbs,cfg,counter):
    parts=[]
    if cfg.working_memory and memory: parts.append("notes you gathered:\n"+"\n".join("- "+m for m in memory))
    if cfg.breadcrumb and crumbs: parts.append("summaries along your path:\n"+"\n".join("- "+clip(c,200) for c in crumbs[-8:]))
    if evidence:
        ev="\n\n".join(f"[{e.metadata.get('source_file') or e.path or e.name}] {clip(e.content or e.summary,1200)}"
                       for e in evidence[:MAX_EVIDENCE])
        parts.append("evidence:\n"+ev)
    ctx="\n\n".join(parts) or "(no context gathered)"
    prompt=(f"answer the question using only the gathered information; be specific.\n\n"
            f"QUESTION: {q.stem}\n\n{ctx}\n\n"
            "give the answer in 1-3 sentences, and cite the source file in brackets if relevant.")
    return llm(prompt,cfg,counter,num_predict=1024 if cfg.thinking else 320,temperature=0)

_JUDGE_CFG=Config()     # judge runs with reasoning off (judge not applicable for this script)

def _parse_judge(raw):
    score=0.0; reason=""
    m=re.search(r"\{.*\}",raw or "",flags=re.S)
    if m:
        try:
            d=_json.loads(m.group(0)); score=float(d.get("score",0)); reason=str(d.get("reason","")).strip()
        except Exception: pass
    if not reason:                      
        mm=re.search(r"(0?\.\d+|0|1(?:\.0+)?)",raw or "")
        if mm: score=float(mm.group(1))
    return max(0.0,min(1.0,score)), reason

def judge_score(q,response,counter):
    al=_ans_list(q); multi=len(al)>1
    prompt=("you are a fair grader. a student answered an open question in their own words and could not see the "
            "choices. the multiple choice version below has the correct option(s) marked and those are the ground "
            "truth. the correct answer may be ONE OR MORE options.\n\n"
            f"QUESTION: {q.stem}\nOPTIONS:\n{_opts_text(q)}\nCORRECT OPTION(S): {', '.join(al)}\n{_gold_text(q)}\n\n"
            f"STUDENT RESPONSE:\n{response or '(empty)'}\n\n"
            "grade from 0.0 to 1.0 how well the response matches the MEANING of the correct option(s). judge by "
            "meaning not wording. "
            +("when several options are correct, give full credit only if the response conveys ALL of them, and "
              "proportional partial credit for covering some. " if multi else
              "give full or near full credit when it conveys the correct idea even in different words, partial when "
              "incomplete. ")
            +"give low credit when it matches a wrong option or is irrelevant. reply with ONLY json:\n"
            '{"score": <number 0.0 to 1.0>, "reason": "<one concise sentence comparing the response to the correct option(s)>"}')
    return _parse_judge(llm(prompt,_JUDGE_CFG,counter,num_predict=400,temperature=0))

def judge_evidence(q,evidence,counter):
    ev="\n\n".join(f"[{e.metadata.get('source_file') or e.path or e.name}] {clip(e.content or e.summary,1500)}"
                   for e in evidence[:MAX_EVIDENCE]) or "(nothing was retrieved)"
    prompt=("you are checking whether a retrieval system fetched the right information, not whether anyone answered. "
            "below is the correct answer(s) to a question and the text the system retrieved.\n\n"
            f"QUESTION: {q.stem}\nCORRECT ANSWER(S): {', '.join(_ans_list(q))}\n{_gold_text(q)}\n\n"
            f"RETRIEVED TEXT:\n{ev}\n\n"
            "rate from 0.0 to 1.0 how well the retrieved text CONTAINS the information needed to reach the correct "
            "answer(s), whether or not it is phrased as the answer; if several answers are correct, weight by how "
            "many are supported. 1.0 means the needed facts are clearly present, 0.0 means absent. reply with ONLY json:\n"
            '{"score": <number 0.0 to 1.0>, "reason": "<one concise sentence>"}')
    return _parse_judge(llm(prompt,_JUDGE_CFG,counter,num_predict=400,temperature=0))

print("eval core ready; configs, virtual subfolders and the agent are defined")

eval core ready; configs, virtual subfolders and the agent are defined


In [5]:
if not TREE_FILE.exists():
    raise SystemExit(f"tree not found at {TREE_FILE.resolve()}; build it first with prototype 5")
ROOT = TreeNode.from_dict(json.loads(TREE_FILE.read_text(encoding="utf-8")))
print(f"loaded tree {ROOT.name}; {ROOT.count_leaves()} leaves and {len(ROOT.children)} top level children")

loaded tree folders; 95458 leaves and 10 top level children


In [ ]:
import json, time, re
from pathlib import Path
import pandas as pd
try:
    from tqdm.auto import tqdm as _tqdm          
except Exception:
    _tqdm=None                                   

def _blank(v):
    if v is None: return True
    s=str(v).strip(); return s=="" or s.lower()=="nan"

def _parse_options_cell(text):
    opts={}
    for line in re.split(r"[\r\n;]+", str(text)):
        mm=re.match(r"^\s*\(?\s*([A-Fa-f])\s*[).:\-\u2013]\s*(.+)$", line.strip())
        if mm and mm.group(1).upper() not in opts: opts[mm.group(1).upper()]=mm.group(2).strip()
    return opts

def _parse_answers(cell):
    found={m.upper() for m in re.findall(r"(?<![A-Za-z])([A-Fa-f])(?![A-Za-z])", str(cell))}
    return [L for L in "ABCDEF" if L in found]

def _match(df):
    norm={re.sub(r"[^a-z]","",str(c).lower()):c for c in df.columns}
    def col(*names):
        for n in names:
            if n in norm: return norm[n]
        return None
    return dict(
        q=col("question","q","prompt","stem"),
        ans=col("correctanswer","answer","correct","gold","label","key"),
        idc=col("number","id","qid","index"),
        diff=col("difficulty","level"),
        combined=col("mcoptions","options","choices","mcq","multiplechoiceoptions","mcoptionsabcd"),
        A=col("a","optiona","choicea"), B=col("b","optionb","choiceb"),
        C=col("c","optionc","choicec"), D=col("d","optiond","choiced"), E=col("e","optione","choicee"), F=col("f","optionf","choicef"))

def _find_header(raw):
    best,score=0,-1
    for r in range(min(8,len(raw))):
        cells=[re.sub(r"[^a-z]","",str(x).lower()) for x in raw.iloc[r].tolist()]
        s=sum(any(k in c for k in ("question","answer","option","number","choice")) for c in cells)
        if s>score: score,best=s,r
    return best

def _prep(raw):
    hr=_find_header(raw)
    df=raw.iloc[hr+1:].copy(); df.columns=[str(c) for c in raw.iloc[hr].tolist()]
    return df.reset_index(drop=True)

def _rows_from_df(df, sheet, out, seen):
    m=_match(df)
    has_split=all(m[k] for k in ("A","B","C","D"))
    if not (m["q"] and m["ans"] and (has_split or m["combined"])):
        print(f"  skip sheet '{sheet}'; columns found were {list(df.columns)}"); return

    def add(qid,stem,opts,ansv,diffv):
        opts={L:str(opts[L]).strip() for L in "ABCDEF" if L in opts and not _blank(opts.get(L))}
        if _blank(stem) or len(opts)<2: return False       
        answers=[a for a in _parse_answers(ansv) if a in opts] or _parse_answers(ansv)
        if not answers: return False                        
        qid=str(qid).strip() if not _blank(qid) else f"{sheet}_{len(out)+1}"
        while qid in seen: qid+="_x"                       
        seen.add(qid)
        out.append(Question(qid,str(stem).strip(),opts,answers[0],
                            "" if _blank(diffv) else str(diffv).strip(), answers))
        return True

    kept=0
    if has_split:
        for _,r in df.iterrows():
            opts={L:r[m[L]] for L in "ABCDEF" if m.get(L)}
            kept+=add(r[m["idc"]] if m["idc"] else None, r[m["q"]], opts, r[m["ans"]],
                      r[m["diff"]] if m["diff"] else None)
    else:
        keycol=m["idc"] or m["q"]                         
        group=df[keycol].apply(lambda x: not _blank(x)).cumsum()
        for _,sub in df.groupby(group):
            def first(c):                                 
                for v in sub[c].tolist():
                    if not _blank(v): return v
                return None
            lines="\n".join(str(v) for v in sub[m["combined"]].tolist() if not _blank(v))
            opts=_parse_options_cell(lines)
            kept+=add(first(m["idc"]) if m["idc"] else None, first(m["q"]), opts, first(m["ans"]),
                      first(m["diff"]) if m["diff"] else None)
    print(f"  sheet '{sheet}': kept {kept}")

# read the question file; reads every tab of an xlsx and matches columns loosely
def load_questions(path, exclude_words=("which",)):
    p=Path(path); ext=p.suffix.lower(); out=[]; seen=set()
    if ext in (".xlsx",".xls"):
        for name,raw in pd.read_excel(p, sheet_name=None, header=None).items():   # every tab, raw
            _rows_from_df(_prep(raw), name, out, seen)
    elif ext==".tsv":
        _rows_from_df(_prep(pd.read_csv(p,sep="\t",header=None,dtype=object)), "tsv", out, seen)
    else:
        _rows_from_df(_prep(pd.read_csv(p,header=None,dtype=object)), "csv", out, seen)
    if exclude_words:                                  # drop any stem containing one of these words
        ban=[w.lower() for w in exclude_words]
        before=len(out)
        out=[q for q in out if not any(w in q.stem.lower() for w in ban)]
        if before-len(out): print(f"excluded {before-len(out)} questions containing {list(exclude_words)}")
    if not out:
        raise ValueError("no valid questions parsed; need question, options and a correct answer letter")
    print(f"parsed {len(out)} questions total from {p.name}")
    return out


EVAL_CACHE_DIR=Path("eval_cache"); RESULTS_FILE=EVAL_CACHE_DIR/"results.json"
CACHE={}

def load_cache():
    global CACHE
    if RESULTS_FILE.exists():
        CACHE=json.loads(RESULTS_FILE.read_text())
    print(f"results cache: {len(CACHE)} stored runs")

def save_cache():
    EVAL_CACHE_DIR.mkdir(exist_ok=True)
    tmp=RESULTS_FILE.with_suffix(".tmp")          # write then swap so a crash mid write cant corrupt it
    tmp.write_text(json.dumps(CACHE)); tmp.replace(RESULTS_FILE)

class _PlainBar:
    def __init__(self,total,initial,desc): self.t=total; self.n=initial; self.d=desc
    def update(self,k=1): self.n+=k; print(f"\r{self.d} {self.n}/{self.t}",end="",flush=True)
    def set_postfix(self,**k): pass
    def set_description(self,d): self.d=d
    def close(self): print()

def _bar(total,initial,desc):
    if _tqdm is not None:
        return _tqdm(total=total,initial=initial,unit="q",desc=desc,dynamic_ncols=True)
    return _PlainBar(total,initial,desc)

import random as _random
from dataclasses import replace as _replace

# answer one question under one config, then score it; judge mode grades a free response 0-1 with a reason
def _eval_one(cfg,q,live=False):
    c=Counters(); t0=time.perf_counter()
    res=run_agent(q,cfg,c,trace=True,live=live); dt=time.perf_counter()-t0
    if SCORE_MODE=="letter":
        score=float(res["response"]==q.answer); reason=""
    else:
        score,reason=judge_score(q,res["response"],Counters())   # judge tokens kept out of the config totals
    srcs=[]
    for e in res["evidence"]:
        s=e.metadata.get("source_file") or e.path or e.name
        if s and s not in srcs: srcs.append(s)
    return {"config":config_name(cfg),"key":config_key(cfg),"qid":q.qid,
            "response":res["response"],"gold":q.answer,"score":round(score,3),"judge":reason,
            "correct":int(score>=0.5),"time":round(dt,3),
            "in_tok":c.in_tok,"out_tok":c.out_tok,"calls":c.calls,
            "steps":res["steps"],"backtracks":res["backtracks"],
            "path":res["path"],"sources":", ".join(srcs[:3]),"difficulty":q.difficulty}

# one config over the questions; cached per config and question so its crash resumable
def run_config(cfg,questions,max_q=None,verbose=False):
    reset_vcaches(); qs=questions[:max_q] if max_q else questions; rows=[]; key=config_key(cfg)
    for q in qs:
        ck=f"{key}::{q.qid}"
        if ck in CACHE: rec=CACHE[ck]
        else: rec=_eval_one(cfg,q); CACHE[ck]=rec; save_cache()
        rows.append(rec)
    return rows

# the question and choices and answer block, then the full response, then the judge with its reason
def _print_diag(q,rec):
    o=q.options
    print("  "+"─"*86)
    print(f"  question: {q.stem}")
    for L in "ABCDEF":
        if L in o: print(f"    {L}. {o[L]}")
    print(f"  correct: {', '.join(_ans_list(q))}")
    print()
    print("  agent response:")
    print("    "+(rec["response"] or "(empty)").replace("\n","\n    "))
    print(f"  files reached: {rec['sources'] or '— none —'}")
    print(f"  full path: {rec['path']}  ({rec['steps']} steps, {rec['backtracks']} backtracks)")
    print()
    print(f"  judge: {rec['score']:.2f} — {rec.get('judge') or 'no justification returned'}")

# run n random questions under one config; live path, then the full readout, judge scored
def run_sample(questions, n=2, cfg=None, seed=0):
    cfg=_replace(cfg or Config(), nav_includes_options=False)    # the agent must not see the choices
    reset_vcaches()
    sample=_random.Random(seed).sample(questions, min(n,len(questions)))
    print(f"sampling {len(sample)} questions under: {config_name(cfg)}")
    rows=[]; tot=0.0
    for idx,q in enumerate(sample,1):
        print("\n"+"="*90)
        print(f"[{idx}/{len(sample)}]  {q.qid}")
        rec=_eval_one(cfg,q,live=True)
        rows.append(rec); tot+=rec["score"]
        _print_diag(q,rec)
    print("\n"+"="*90+f"\nmean judge score over {len(sample)} questions: {tot/len(sample):.3f}")
    return rows

# does the saved tree keep raw chunk text or only summaries; a summary-only tree caps accuracy no matter the config
def check_leaf_content(root=None):
    root=root or ROOT
    leaves=[]; stack=[root]
    while stack:
        n=stack.pop()
        if n.node_type=="chunk": leaves.append(n)
        else: stack.extend(n.children)
    total=max(len(leaves),1)
    withc=[len((l.content or "").strip()) for l in leaves if (l.content or "").strip()]
    frac=len(withc)/total
    print(f"leaf nodes: {len(leaves)}")
    print(f"  with raw content: {len(withc)} ({100*frac:.0f}%) | summary only: {len(leaves)-len(withc)}")
    if withc:
        withc.sort(); print(f"  raw content length: median {withc[len(withc)//2]} chars, max {max(withc)}")
    if frac<0.2:
        print("VERDICT: leaves are essentially summary-only, so the agent answers from lossy summaries; "
              "this likely caps accuracy regardless of navigation. rebuilding the tree to keep raw chunk text would help most.")
    elif frac<0.8:
        print("VERDICT: only some leaves carry raw text; check whether the docs you actually query are among the ones missing it.")
    else:
        print("VERDICT: leaves carry raw text, so the evidence is there; low accuracy points at navigation or the answer step, not the tree.")
    return {"leaves":len(leaves),"with_content":len(withc),"fraction":frac}

# grade RETRIEVAL separately from ANSWERING so you can tell which one is costing you accuracy
def run_diagnosis(questions, n=30, cfg=None, seed=0, verbose=True):
    cfg=_replace(cfg or Config(), nav_includes_options=False)
    reset_vcaches()
    sample=_random.Random(seed).sample(questions, min(n,len(questions)))
    print(f"diagnosis on {len(sample)} questions under: {config_name(cfg)}\n"+"-"*78)
    rows=[]; ev_sum=0.0; rp_sum=0.0; miss_ret=0; miss_ans=0; ok=0
    for q in sample:
        res=run_agent(q,cfg,Counters(),trace=False,live=False)
        es,er=judge_evidence(q,res["evidence"],Counters())
        rs,rr=judge_score(q,res["response"],Counters())
        if es<0.5: tag="RETRIEVAL MISS"; miss_ret+=1     # never reached the right info
        elif rs<0.5: tag="ANSWER MISS"; miss_ans+=1       # had the info but answered wrong
        else: tag="ok"; ok+=1
        ev_sum+=es; rp_sum+=rs
        srcs=", ".join(dict.fromkeys(e.metadata.get("source_file") or e.path or e.name for e in res["evidence"]))
        rows.append({"qid":q.qid,"evidence":round(es,3),"response":round(rs,3),"tag":tag,
                     "files":srcs,"resp":res["response"],"ev_reason":er,"rp_reason":rr})
        if verbose:
            print(f"  {q.qid:12s} evidence={es:.2f}  response={rs:.2f}  -> {tag:14s}  {srcs[:54] or '— none —'}")
    k=len(sample)
    print("-"*78)
    print(f"mean evidence-score: {ev_sum/k:.3f}    mean response-score: {rp_sum/k:.3f}")
    print(f"retrieval misses: {miss_ret}/{k} ({100*miss_ret/k:.0f}%)   "
          f"answer misses: {miss_ans}/{k} ({100*miss_ans/k:.0f}%)   ok: {ok}/{k} ({100*ok/k:.0f}%)")
    if miss_ret/k>=0.4:
        print("VERDICT: navigation is the bottleneck, the agent often never reaches the right document. "
              "beam or top-k navigation, query expansion, or best-first search are the things to build next.")
    elif miss_ans/k>=0.4:
        print("VERDICT: retrieval is mostly fine but the final answer step loses it; fix the answer prompt, not the traversal.")
    else:
        print("VERDICT: most questions are handled well; if the headline number still looks low, hand-check the judge, it may be harsh.")
    return rows

# run the whole grid with a progress bar and eta; rerun after a crash and it resumes from the cache
def run_grid(grid,questions,max_q=None,verbose=False):
    qs=questions[:max_q] if max_q else questions
    tasks=[(cfg,q) for cfg in grid for q in qs]
    done=sum(1 for cfg,q in tasks if f"{config_key(cfg)}::{q.qid}" in CACHE)   # already in the cache
    if done: print(f"resuming; {done}/{len(tasks)} runs already cached, the bar starts from there")
    bar=_bar(len(tasks),done,"eval")
    all_rows=[]; score_sum=0.0; n=0; last=None
    for cfg,q in tasks:
        if cfg is not last:
            reset_vcaches(); last=cfg                  # fresh group summaries so tokens stay per config
            bar.set_description((config_name(cfg)[:34]))
        ck=f"{config_key(cfg)}::{q.qid}"
        if ck in CACHE:
            rec=CACHE[ck]                              # resumed; already counted in the bars start
        else:
            rec=_eval_one(cfg,q); CACHE[ck]=rec; save_cache(); bar.update(1)
        all_rows.append(rec); score_sum+=rec["score"]; n+=1
        bar.set_postfix(score=f"{score_sum/n:.2f}",last=f"{rec['score']}")
    bar.close()
    print(f"done; {n} results, overall mean score {score_sum/n:.3f} across all configs")
    return all_rows


# collapse the per question rows into one line per config with means and spreads
def leaderboard(all_rows):
    df=pd.DataFrame(all_rows)
    g=df.groupby(["config","key"],sort=False)
    lb=g.agg(n=("score","size"),correct=("correct","sum"),
             accuracy=("score","mean"),
             mean_time=("time","mean"),std_time=("time","std"),
             mean_in=("in_tok","mean"),std_in=("in_tok","std"),
             mean_out=("out_tok","mean"),std_out=("out_tok","std"),
             total_in=("in_tok","sum"),total_out=("out_tok","sum"),
             mean_calls=("calls","mean")).reset_index()
    lb["accuracy_pct"]=(lb["accuracy"]*100).round(1)
    for c in ["mean_time","std_time","mean_in","std_in","mean_out","std_out","mean_calls"]:
        lb[c]=lb[c].fillna(0).round(2)
    return lb

def by_accuracy(lb):
    return lb.sort_values(["accuracy","mean_time"],ascending=[False,True]).reset_index(drop=True)

def by_speed(lb):
    return lb.sort_values(["mean_time","accuracy"],ascending=[True,False]).reset_index(drop=True)

# write the leaderboard and every individual run so nothing is lost
def save_outputs(lb,all_rows,outdir="eval_cache"):
    d=Path(outdir); d.mkdir(exist_ok=True)
    cols=["config","accuracy_pct","correct","n","mean_time","std_time",
          "mean_in","mean_out","total_in","total_out","mean_calls","key"]
    by_accuracy(lb)[cols].to_csv(d/"leaderboard_by_accuracy.csv",index=False)
    by_speed(lb)[cols].to_csv(d/"leaderboard_by_speed.csv",index=False)
    pd.DataFrame(all_rows).to_csv(d/"per_question.csv",index=False)
    print(f"wrote {d/'leaderboard_by_accuracy.csv'}, leaderboard_by_speed.csv and per_question.csv")

print("eval harness ready; load_questions, run_grid, leaderboard, by_accuracy, by_speed")

eval harness ready; load_questions, run_grid, leaderboard, by_accuracy, by_speed


/opt/homebrew/Cellar/jupyterlab/4.5.7_1/libexec/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import heapq
from dataclasses import dataclass, asdict

# all navigators share one primitive: score every child 0-1 for how likely it leads to the answer
def _score_children(query,node,children,cfg,counter,temp=0.0):
    listing="\n".join(f"[{i}] {c.name}: {clip(c.summary,300)}" for i,c in enumerate(children))
    prompt=("you are searching a tree of document summaries to find the information that answers a question. "
            "rate each option from 0.0 to 1.0 by how likely it contains or leads to that information.\n\n"
            f"QUESTION: {query}\nYOU ARE AT: {node.name}\nOPTIONS:\n{listing}\n\n"
            f"reply with ONLY a json list of {len(children)} numbers in order, like [0.1, 0.8, 0.2].")
    raw=llm(prompt,cfg,counter,num_predict=1200 if cfg.thinking else 512,temperature=temp)
    m=re.search(r"\[[^\]]*\]",raw or "")
    scores=[]
    if m:
        try: scores=[float(x) for x in _json.loads(m.group(0))]
        except Exception: scores=[]
    if len(scores)!=len(children): scores=[0.5]*len(children)   # parse failed; treat all equal
    return [max(0.0,min(1.0,s)) for s in scores]

# rewrite the query into salient terms so it matches summary vocabulary better
def _expand_query(q,counter):
    prompt=("rewrite this question into a compact set of salient search terms plus a one sentence paraphrase, "
            "to help locate the relevant document. do not answer it.\n\nQUESTION: "+q.stem+
            "\n\nreply with ONLY the expanded query text.")
    return q.stem+"\n"+llm(prompt,Config(),counter,num_predict=140,temperature=0)

def _is_leaf(n): return n.is_leaf() or not n.children

# beam search: keep the best B partial paths at each level and pool the leaves they reach
def nav_beam(query,cfg,counter):
    B=max(1,cfg.beam_width); frontier=[ROOT]; visited={ROOT.node_id}; evidence=[]; steps=0
    while frontier and steps<cfg.max_steps and len(evidence)<cfg.max_leaves:
        steps+=1; scored=[]
        for node in frontier:
            kids=[c for c in get_nav_children(node,cfg,counter) if c.node_id not in visited]
            if not kids: continue
            for s,c in zip(_score_children(query,node,kids,cfg,counter),kids): scored.append((s,c))
        if not scored: break
        scored.sort(key=lambda x:-x[0]); nextf=[]
        for s,c in scored[:B]:
            visited.add(c.node_id)
            if _is_leaf(c) or not get_nav_children(c,cfg,counter): evidence.append(c)
            else: nextf.append(c)
        frontier=nextf
    return evidence[:cfg.max_leaves], f"beam w{B}, {len(evidence)} leaves", steps

# best first: a global priority queue, always expand the most promising node next
def nav_bestfirst(query,cfg,counter):
    pq=[(-1.0,0,ROOT)]; visited={ROOT.node_id}; evidence=[]; exp=0; tie=0
    while pq and exp<cfg.frontier_budget and len(evidence)<cfg.max_leaves:
        _,_,node=heapq.heappop(pq)
        kids=[c for c in get_nav_children(node,cfg,counter) if c.node_id not in visited]
        if not kids:
            if _is_leaf(node): evidence.append(node)
            continue
        exp+=1
        for s,c in zip(_score_children(query,node,kids,cfg,counter),kids):
            visited.add(c.node_id); tie+=1; heapq.heappush(pq,(-s,tie,c))
    while pq and len(evidence)<cfg.max_leaves:                 # drain the best remaining leaves
        _,_,node=heapq.heappop(pq)
        if _is_leaf(node) or not get_nav_children(node,cfg,counter): evidence.append(node)
    return evidence[:cfg.max_leaves], f"bestfirst, {exp} expansions", exp

def _greedy_path(query,cfg,counter,temp):
    node=ROOT; visited={ROOT.node_id}; d=0
    while d<cfg.max_steps:
        d+=1
        kids=[c for c in get_nav_children(node,cfg,counter) if c.node_id not in visited]
        if not kids: break
        ss=_score_children(query,node,kids,cfg,counter,temp=temp)
        node=kids[max(range(len(kids)),key=lambda i:ss[i])]; visited.add(node.node_id)
        if _is_leaf(node) or not get_nav_children(node,cfg,counter): return node
    return node if _is_leaf(node) else None

# run the greedy path several times with sampling and pool the distinct leaves; fixes one bad path
def nav_selfconsistency(query,cfg,counter):
    evidence=[]; seen=set()
    for r in range(max(1,cfg.sc_runs)):
        leaf=_greedy_path(query,cfg,counter,temp=0.0 if r==0 else 0.7)
        if leaf and leaf.node_id not in seen: seen.add(leaf.node_id); evidence.append(leaf)
        if len(evidence)>=cfg.max_leaves: break
    return evidence, f"selfconsistency, {cfg.sc_runs} runs", cfg.sc_runs


@dataclass(frozen=True)
class NavCfg:
    navigator:str="beam"        # baseline greedy beam bestfirst selfconsistency
    strategy:str="cluster"
    max_branch:int=6
    group_summary:str="heuristic"
    thinking:bool=False
    beam_width:int=3
    frontier_budget:int=12
    sc_runs:int=3
    query_expansion:bool=False
    max_leaves:int=5
    max_steps:int=24
    decision_cap:int=512        # only so the llm helper never reaches for a missing attr

def navcfg_name(cfg):
    b=[cfg.navigator]
    if cfg.navigator=="beam": b.append(f"w{cfg.beam_width}")
    if cfg.navigator=="bestfirst": b.append(f"budget{cfg.frontier_budget}")
    if cfg.navigator=="selfconsistency": b.append(f"runs{cfg.sc_runs}")
    if cfg.query_expansion: b.append("qexp")
    if cfg.strategy!="cluster": b.append(cfg.strategy)
    if cfg.max_leaves!=5: b.append(f"L{cfg.max_leaves}")
    return " ".join(b)

def navcfg_key(cfg): return hashlib.md5(repr(asdict(cfg)).encode()).hexdigest()[:12]

# pick a navigator and return the gathered evidence; baseline is the current greedy memwalker agent
def navigate(q,cfg,counter):
    if cfg.navigator=="baseline":
        bc=Config(strategy=cfg.strategy,max_branch=cfg.max_branch,group_summary=cfg.group_summary,thinking=cfg.thinking)
        res=run_agent(q,bc,counter,trace=False,live=False)
        return res["evidence"], res["path"], res["steps"]
    nq=_expand_query(q,counter) if cfg.query_expansion else q.stem
    return {"beam":nav_beam,"bestfirst":nav_bestfirst,"selfconsistency":nav_selfconsistency}[cfg.navigator](nq,cfg,counter)

# navigate, write an answer from the pooled evidence, then grade retrieval and answering separately
def evaluate_nav(cfg,q):
    c=Counters(); t0=time.perf_counter()
    evidence,trail,steps=navigate(q,cfg,c)
    response=_answer_response(q,[],evidence,[],Config(thinking=cfg.thinking),c)
    dt=time.perf_counter()-t0
    es,er=judge_evidence(q,evidence,Counters())
    rs,rr=judge_score(q,response,Counters())
    srcs=[]
    for e in evidence:
        s=e.metadata.get("source_file") or e.path or e.name
        if s and s not in srcs: srcs.append(s)
    return {"nav":navcfg_name(cfg),"key":navcfg_key(cfg),"qid":q.qid,
            "evidence_score":round(es,3),"response_score":round(rs,3),"leaves":len(evidence),
            "steps":steps,"time":round(dt,3),"in_tok":c.in_tok,"out_tok":c.out_tok,"calls":c.calls,
            "files":", ".join(srcs[:3]),"response":response,"ev_reason":er,"rp_reason":rr}

# the navigators to compare; baseline is the current system that scored 16% retrieval
def default_navgrid():
    return [NavCfg(navigator="baseline"),
            NavCfg(navigator="beam",beam_width=2),
            NavCfg(navigator="beam",beam_width=3),
            NavCfg(navigator="beam",beam_width=3,query_expansion=True),
            NavCfg(navigator="beam",beam_width=3,max_leaves=8),
            NavCfg(navigator="bestfirst",frontier_budget=12),
            NavCfg(navigator="selfconsistency",sc_runs=3)]


NAV_CACHE_DIR=Path("nav_cache"); NAV_RESULTS=NAV_CACHE_DIR/"results.json"; NAVCACHE={}

def nav_load_cache():
    global NAVCACHE
    if NAV_RESULTS.exists(): NAVCACHE=json.loads(NAV_RESULTS.read_text())
    print(f"nav cache: {len(NAVCACHE)} stored runs")

def nav_save_cache():
    NAV_CACHE_DIR.mkdir(exist_ok=True)
    tmp=NAV_RESULTS.with_suffix(".tmp"); tmp.write_text(json.dumps(NAVCACHE)); tmp.replace(NAV_RESULTS)

# compare navigators on the questions; n=None uses the full set, primary metric is evidence-score
def run_navsweep(navgrid, questions, n=None, seed=0):
    sample=questions if not n else _random.Random(seed).sample(questions, min(n,len(questions)))
    tasks=[(cfg,q) for cfg in navgrid for q in sample]
    done=sum(1 for cfg,q in tasks if f"{navcfg_key(cfg)}::{q.qid}" in NAVCACHE)
    if done: print(f"resuming; {done}/{len(tasks)} already cached")
    bar=_bar(len(tasks),done,"nav"); rows=[]; last=None
    for cfg,q in tasks:
        if cfg is not last: reset_vcaches(); last=cfg; bar.set_description(navcfg_name(cfg)[:30])
        ck=f"{navcfg_key(cfg)}::{q.qid}"
        if ck in NAVCACHE: rec=NAVCACHE[ck]
        else: rec=evaluate_nav(cfg,q); NAVCACHE[ck]=rec; nav_save_cache(); bar.update(1)
        rows.append(rec)
        bar.set_postfix(ev=f"{np.mean([r['evidence_score'] for r in rows]):.2f}")
    bar.close()
    return rows

def nav_leaderboard(rows):
    df=pd.DataFrame(rows); g=df.groupby(["nav","key"],sort=False)
    lb=g.agg(n=("evidence_score","size"),
             evidence=("evidence_score","mean"),ev_std=("evidence_score","std"),
             response=("response_score","mean"),
             leaves=("leaves","mean"),mean_time=("time","mean"),
             mean_in=("in_tok","mean"),mean_out=("out_tok","mean")).reset_index()
    lb["ev_sem"]=(lb["ev_std"]/np.sqrt(lb["n"])).round(3)        # standard error; gaps under ~2x this are noise
    for c in ["evidence","response","leaves","mean_time","mean_in","mean_out"]: lb[c]=lb[c].round(3)
    return lb.sort_values("evidence",ascending=False).reset_index(drop=True)

def save_nav_outputs(lb,rows):
    NAV_CACHE_DIR.mkdir(exist_ok=True)
    lb.to_csv(NAV_CACHE_DIR/"nav_leaderboard.csv",index=False)
    pd.DataFrame(rows).to_csv(NAV_CACHE_DIR/"nav_per_question.csv",index=False)
    print(f"wrote {NAV_CACHE_DIR/'nav_leaderboard.csv'} and nav_per_question.csv")

print("nav mechanisms ready; default_navgrid, run_navsweep, nav_leaderboard")

nav mechanisms ready; default_navgrid, run_navsweep, nav_leaderboard


In [8]:
# the treerag agent to use; beam width 2 over cluster subfolders
AGENT = NavCfg(navigator="beam", beam_width=2)

ANS_DIR=Path("answers_cache"); ANS_RESULTS=ANS_DIR/"answers.json"; ANSWERS={}
def ans_load():
    global ANSWERS
    if ANS_RESULTS.exists(): ANSWERS=json.loads(ANS_RESULTS.read_text())
    print(f"answers cache: {len(ANSWERS)} stored")
def ans_save():
    ANS_DIR.mkdir(exist_ok=True)
    tmp=ANS_RESULTS.with_suffix(".tmp"); tmp.write_text(json.dumps(ANSWERS)); tmp.replace(ANS_RESULTS)

# navigate with the agent, write a response from the evidence, and record everything for one question
def answer_one(q, cfg):
    c=Counters(); t0=time.perf_counter()
    evidence,path,steps=navigate(q,cfg,c)
    response=_answer_response(q,[],evidence,[],Config(thinking=cfg.thinking),c)
    dt=time.perf_counter()-t0
    files=[]
    for e in evidence:
        s=e.metadata.get("source_file") or e.path or e.name
        if s and s not in files: files.append(s)
    return {"id":q.qid,"question":q.stem,"options":q.options,"correct_answers":_ans_list(q),
            "treerag_response":response,"files_referenced":files,"path":path,
            "leaves":len(evidence),"steps":steps,"time_sec":round(dt,3),
            "in_tokens":c.in_tok,"out_tokens":c.out_tok}

# run every question through the agent and save the responses; resumable
def run_all(questions, cfg, out_file):
    ans_load()
    todo=[q for q in questions if q.qid not in ANSWERS]
    print(f"{len(questions)} questions; {len(questions)-len(todo)} already answered, {len(todo)} to go")
    bar=_bar(len(questions),len(questions)-len(todo),"answering")
    for q in todo:
        ANSWERS[q.qid]=answer_one(q,cfg); ans_save(); bar.update(1)
    bar.close()
    records=[ANSWERS[q.qid] for q in questions]               # keep the original question order
    Path(out_file).write_text(json.dumps(records,indent=2))
    print(f"\nwrote {len(records)} responses to {out_file}")
    return records

In [9]:
# load questions.json; schema qid/stem/options/answer/answers/difficulty/sheet, shared with qms_search
# tolerant of single-quoted dumps via ast.literal_eval, and of one-or-more correct answers
import ast
def load_questions_json(path):
    raw=Path(path).read_text(encoding="utf-8")
    try: data=json.loads(raw)
    except Exception: data=ast.literal_eval(raw)
    if isinstance(data,dict): data=data.get("questions",list(data.values()))
    out=[]; skipped=[]
    for i,d in enumerate(data,1):
        qid=str(d.get("qid") or f"q{i}")
        stem=str(d.get("stem") or d.get("question") or "").strip()
        opts={str(k).upper():str(v).strip() for k,v in (d.get("options") or {}).items() if str(v).strip()}
        ans=d.get("answers") or ([d.get("answer")] if d.get("answer") else [])
        ans=[m.upper() for m in re.findall(r"(?<![A-Za-z])([A-Fa-f])(?![A-Za-z])", " ".join(map(str,ans)))]
        ans=[L for L in "ABCDEF" if L in set(ans)]
        if not stem:           skipped.append((qid,"no stem",            (d.get("stem") or d.get("question")))); continue
        if len(opts)<2:        skipped.append((qid,"fewer than 2 options",d.get("options"))); continue
        if not ans:            skipped.append((qid,"no answer letter",    d.get("answers") or d.get("answer"))); continue
        out.append(Question(qid, stem, opts, ans[0], str(d.get("difficulty") or ""), ans))
    return out, skipped

questions, skipped = load_questions_json(QUESTIONS_FILE)
print(f"loaded {len(questions)} questions from {QUESTIONS_FILE}; skipped {len(skipped)}")
for qid,reason,val in skipped:
    print(f"  SKIP [{qid}] {reason}: {str(val)[:90]}")
ans_load()

loaded 325 questions from questions_updated.json; skipped 0
answers cache: 2 stored


In [10]:
# run every question through beam-w2 TreeRAG and write treerag_answers.json
records = run_all(questions, AGENT, ANSWERS_FILE)

import pandas as pd
df=pd.DataFrame(records)
print("\nsample of the output:")
print(df[["id","files_referenced","time_sec","in_tokens","out_tokens"]].head(8).to_string(index=False))
print(f"\nmean time/question: {df['time_sec'].mean():.1f}s   mean leaves: {df['leaves'].mean():.1f}")
df.head()

answers cache: 2 stored
325 questions; 2 already answered, 323 to go


answering:   6%|██████                                                                                        | 21/325 [40:20<8:00:27, 94.83s/q]

[waiting for ollama] ReadTimeout: timed out; retrying
[waiting for ollama] ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download; retrying
[waiting for ollama] ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download; retrying
[waiting for ollama] ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download; retrying
[waiting for ollama] ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download; retrying
[waiting for ollama] ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download; retrying
[waiting for ollama] ConnectionError: Failed to connect to Ollama. Please check that Ollama is downlo

answering: 100%|█| 325/325 [14:06:45<00:00, 15                                                                                                  



wrote 325 responses to treerag_answers.json

sample of the output:
   id                                                                                                                                                                                   files_referenced  time_sec  in_tokens  out_tokens
QMS-1                  [folders/Assay_Validation/Validation Supporting Documents/Informatics Validation.docx, folders/Assay_Validation/Validation Supporting Documents/CAP validation requirements.docx]    92.903       5193        3566
QMS-2                                                             [folders/Technical SOPs and Worksheets/Initial Sample QC - Qubit.docx, folders/Quality SOPs and Worksheets/Document Control Plan.docx]   253.292       7032        4726
QMS-3                                            [folders/CAPA/2024-01-04 CAPA Form - FFPE extracted with FF protocol.docx, folders/Quality SOPs and Worksheets/Non-Conformance and CAPA Procedure.docx]   137.536       7251        4

,id,question,options,correct_answers,treerag_response,files_referenced,path,leaves,steps,time_sec,in_tokens,out_tokens
0,QMS-1,Where do you find SOPs?,"{'A': 'In a central lab binder', 'B': 'On the ...",[B],SOPs are located in the **folders/Assay_Valida...,[folders/Assay_Validation/Validation Supportin...,"beam w2, 3 leaves",3,7,92.903,5193,3566
1,QMS-2,What version of an SOP should be used?,"{'A': 'Version 1.0 should always be used', 'B'...",[C],"The SOP that is used must be the **current, ma...",[folders/Technical SOPs and Worksheets/Initial...,"beam w2, 3 leaves",3,10,253.292,7032,4726
2,QMS-3,How do you initiate the CAPA process?,{'A': 'Complete the CAPA form on the Quality S...,[D],"To start a CAPA, staff first report the identi...",[folders/CAPA/2024-01-04 CAPA Form - FFPE extr...,"beam w2, 3 leaves",3,9,137.536,7251,4798
3,QMS-4,What conditions count as a non-conformance?,{'A': 'Test results are at the high end of the...,[A],A non‑conformance exists whenever a deviation ...,[folders/Quality SOPs and Worksheets/Non-Confo...,"beam w2, 3 leaves",3,11,176.425,8332,5744
4,QMS-5,What is the main purpose of Proficiency Testin...,{'A': 'To demonstrate that an assay is perform...,[A],The primary purpose of Proficiency Testing (PT...,[folders/Management/Proficiency Testing/Sample...,"beam w2, 2 leaves",2,10,137.824,9583,5646
